[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/01_collection/A6_community_map_import.ipynb)

# A6: Community Map Import (Gellerman Data)

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Parse KML files** (Google Earth/Maps format) using Python
2. **Extract structured data from HTML** embedded in KML descriptions
3. **Categorize URLs** by domain (news source identification)
4. **Validate geographic coordinates** against city boundaries

## Why This Matters

Community members often maintain more comprehensive data than official sources:
- **Eric Gellerman** tracks ~203 Berkeley housing projects on Google My Maps
- He includes **news coverage links** (Berkeleyside, SFYimby, SF Chronicle)
- His map captures **proposed projects** before they enter the permit system

Integrating community data with official data creates a more complete picture.

---

## Overview

Import housing project data from Eric Gellerman's Berkeley Development Map (Google My Maps).

**Input:**
- `data/reference/gellerman_berkeley_housing.kml` (or fetched from Google Maps)

**Outputs:**
- `outputs/gellerman_raw.csv` - All projects with coordinates
- `outputs/gellerman_news_links.csv` - Extracted news URLs
- `outputs/a6_import_summary.txt` - Import statistics

---

## 1. Setup & Environment

In [1]:
# ============================================================================
# COLAB ENVIRONMENT SETUP
# ============================================================================

import os
import sys
from pathlib import Path

print('Setting up environment...')
print('='*70)

# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

if IN_COLAB:
    repo_path = Path('/content/berkeley-housing-analysis')
    
    if not repo_path.exists():
        print('\nCloning repository...')
        !git clone https://github.com/blockXblock/berkeley-housing-analysis.git
        print('Repository cloned')
    else:
        print('\nRepository already exists')
        !cd /content/berkeley-housing-analysis && git pull
    
    os.chdir(repo_path)
    
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))
    
    # Create directories
    (repo_path / 'outputs').mkdir(exist_ok=True)
    
    ROOT = repo_path
else:
    # Local environment - find project root
    def find_project_root():
        current = Path.cwd()
        for path in [current] + list(current.parents):
            if (path / '00_config').exists() or (path / 'data').exists():
                return path
        return current
    
    ROOT = find_project_root()
    os.chdir(ROOT)

# Create outputs directory
(ROOT / 'outputs').mkdir(exist_ok=True)

print(f'\nWorking directory: {os.getcwd()}')
print('='*70)

Setting up environment...
Running locally

Working directory: /Users/johngage/berkeley-data


In [2]:
# ============================================================================
# IMPORTS
# ============================================================================

import pandas as pd
import re
import xml.etree.ElementTree as ET
from datetime import datetime
import time
import json

# For fetching KML from network
try:
    import requests
    HAS_REQUESTS = True
except ImportError:
    HAS_REQUESTS = False
    print('Note: requests library not available, will use local KML file only')

print('Imports loaded successfully')

Imports loaded successfully


In [3]:
# ============================================================================
# UTILITY FUNCTIONS - Timestamps and Timing
# ============================================================================

_cell_start_time = None

def timer_start(label=""):
    """Start execution timer"""
    global _cell_start_time
    _cell_start_time = time.time()
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    if label:
        print(f'{label}')
    print(f'Started: {now}')
    print('='*70)

def timer_end():
    """End execution timer and show duration"""
    global _cell_start_time
    duration = time.time() - _cell_start_time
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print('='*70)
    print(f'Completed: {now}')
    print(f'Duration: {duration:.2f} seconds')

print('Timestamp utilities loaded')

Timestamp utilities loaded


## 2. KML Parsing Functions

### Key Concept: KML Format

**KML (Keyhole Markup Language)** is an XML format used by Google Earth/Maps:

```xml
<Placemark>
  <name>Project Name</name>
  <description><![CDATA[HTML content with links]]></description>
  <Point>
    <coordinates>-122.267,37.867,0</coordinates>
  </Point>
</Placemark>
```

**Challenge:** The description field contains raw HTML, which we need to parse for URLs.

In [4]:
# ============================================================================
# KML NAMESPACE
# ============================================================================

# KML uses XML namespaces - we need to specify this for parsing
# Note: Google sometimes uses different namespace URIs
KML_NAMESPACES = {
    'kml': 'http://www.opengis.net/kml/2.2',
    'kml_old': 'http://earth.google.com/kml/2.2'
}

def get_kml_namespace(root):
    """
    Detect the KML namespace from the root element.
    
    Different KML files may use different namespace URIs.
    """
    # Check the root tag for namespace
    if root.tag.startswith('{'):
        ns_end = root.tag.find('}')
        namespace = root.tag[1:ns_end]
        return {'kml': namespace}
    return KML_NAMESPACES

print('KML namespace utilities loaded')

KML namespace utilities loaded


In [5]:
# ============================================================================
# KML PARSING FUNCTIONS
# ============================================================================

def parse_kml_placemarks(kml_content):
    """
    Parse KML content and extract all placemarks.
    
    Parameters:
    -----------
    kml_content : str
        Raw KML XML content
        
    Returns:
    --------
    list of dict
        Each dict contains: name, description, latitude, longitude
    """
    placemarks = []
    
    try:
        # Parse XML
        root = ET.fromstring(kml_content)
        
        # Detect namespace
        ns = get_kml_namespace(root)
        
        # Try different namespace prefixes
        for ns_prefix, ns_uri in ns.items():
            ns_dict = {'kml': ns_uri}
            
            # Find all Placemark elements
            for placemark in root.findall('.//kml:Placemark', ns_dict):
                record = extract_placemark_data(placemark, ns_dict)
                if record:
                    placemarks.append(record)
        
        # Also try without namespace (some KML files don't use namespaces properly)
        if not placemarks:
            for placemark in root.iter('Placemark'):
                record = extract_placemark_data_no_ns(placemark)
                if record:
                    placemarks.append(record)
    
    except ET.ParseError as e:
        print(f'XML Parse Error: {e}')
    except Exception as e:
        print(f'Error parsing KML: {e}')
    
    return placemarks


def extract_placemark_data(placemark, ns):
    """
    Extract data from a single Placemark element (with namespace).
    """
    record = {
        'name': None,
        'description': None,
        'latitude': None,
        'longitude': None
    }
    
    # Get name
    name_elem = placemark.find('kml:name', ns)
    if name_elem is not None and name_elem.text:
        record['name'] = name_elem.text.strip()
    
    # Get description
    desc_elem = placemark.find('kml:description', ns)
    if desc_elem is not None and desc_elem.text:
        record['description'] = desc_elem.text.strip()
    
    # Get coordinates (try Point first, then other geometry types)
    coords = None
    
    # Try Point
    point = placemark.find('kml:Point/kml:coordinates', ns)
    if point is not None and point.text:
        coords = point.text.strip()
    
    # Try Polygon (use first coordinate)
    if not coords:
        polygon = placemark.find('.//kml:coordinates', ns)
        if polygon is not None and polygon.text:
            coords = polygon.text.strip().split()[0]  # First point
    
    # Parse coordinates (lon,lat,altitude format)
    if coords:
        try:
            parts = coords.split(',')
            record['longitude'] = float(parts[0])
            record['latitude'] = float(parts[1])
        except (ValueError, IndexError):
            pass
    
    return record if record['name'] else None


def extract_placemark_data_no_ns(placemark):
    """
    Extract data from a Placemark element (without namespace).
    """
    record = {
        'name': None,
        'description': None,
        'latitude': None,
        'longitude': None
    }
    
    # Get name
    name_elem = placemark.find('name')
    if name_elem is not None and name_elem.text:
        record['name'] = name_elem.text.strip()
    
    # Get description
    desc_elem = placemark.find('description')
    if desc_elem is not None and desc_elem.text:
        record['description'] = desc_elem.text.strip()
    
    # Get coordinates
    coords_elem = placemark.find('.//coordinates')
    if coords_elem is not None and coords_elem.text:
        coords = coords_elem.text.strip().split()[0]  # First point
        try:
            parts = coords.split(',')
            record['longitude'] = float(parts[0])
            record['latitude'] = float(parts[1])
        except (ValueError, IndexError):
            pass
    
    return record if record['name'] else None

print('KML parsing functions loaded')

KML parsing functions loaded


## 3. URL Extraction and Categorization

### Key Concept: Regex for URL Extraction

The description field contains HTML with embedded links:
```html
<a href="https://berkeleyside.org/2024/01/project-news">Article</a>
```

We use regex to find all URLs and then categorize them by source.

In [6]:
# ============================================================================
# URL EXTRACTION FUNCTIONS
# ============================================================================

def extract_urls_from_html(html_content):
    """
    Extract all URLs from HTML content.
    
    Parameters:
    -----------
    html_content : str
        HTML string (from KML description field)
        
    Returns:
    --------
    list of str
        All URLs found in the content
    """
    if not html_content:
        return []
    
    # Regex pattern for URLs
    # Matches http:// and https:// URLs, stopping at whitespace, quotes, or HTML tags
    url_pattern = r'https?://[^\s<>"\']+'  
    
    urls = re.findall(url_pattern, html_content)
    
    # Clean up URLs (remove trailing punctuation)
    cleaned_urls = []
    for url in urls:
        # Remove trailing punctuation that might have been captured
        url = url.rstrip('.,;:)]\'"')
        if url:
            cleaned_urls.append(url)
    
    return list(set(cleaned_urls))  # Remove duplicates


def categorize_url(url):
    """
    Categorize a URL by its source/domain.
    
    Parameters:
    -----------
    url : str
        URL to categorize
        
    Returns:
    --------
    str
        Source category (Berkeleyside, SFYimby, SF Chronicle, etc.)
    """
    url_lower = url.lower()
    
    # News sources (in order of priority)
    if 'berkeleyside.org' in url_lower or 'berkeleyside.com' in url_lower:
        return 'Berkeleyside'
    elif 'sfyimby.com' in url_lower:
        return 'SFYimby'
    elif 'sfchronicle.com' in url_lower:
        return 'SF Chronicle'
    elif 'dailycal.org' in url_lower:
        return 'Daily Cal'
    elif 'sfgate.com' in url_lower:
        return 'SFGate'
    elif 'mercurynews.com' in url_lower:
        return 'Mercury News'
    elif 'eastbaytimes.com' in url_lower:
        return 'East Bay Times'
    
    # City/Government sources
    elif 'cityofberkeley.info' in url_lower or 'berkeleyca.gov' in url_lower:
        return 'City of Berkeley'
    elif 'aca-prod.accela.com' in url_lower:
        return 'Accela Permits'
    
    # Social/Other
    elif 'twitter.com' in url_lower or 'x.com' in url_lower:
        return 'Twitter/X'
    elif 'google.com/maps' in url_lower:
        return 'Google Maps'
    else:
        return 'Other'


def extract_news_links(placemarks):
    """
    Extract all news links from placemarks.
    
    Returns DataFrame with: project_name, url, source_category
    """
    news_data = []
    
    for pm in placemarks:
        urls = extract_urls_from_html(pm.get('description', ''))
        
        for url in urls:
            category = categorize_url(url)
            news_data.append({
                'project_name': pm['name'],
                'latitude': pm.get('latitude'),
                'longitude': pm.get('longitude'),
                'url': url,
                'source_category': category
            })
    
    return pd.DataFrame(news_data)

print('URL extraction functions loaded')

URL extraction functions loaded


## 4. Address Normalization

Apply the same normalization patterns from A2 to enable matching.

In [7]:
# ============================================================================
# ADDRESS NORMALIZATION (from A2 patterns)
# ============================================================================

def normalize_address(address):
    """
    Normalize an address for matching.
    
    Follows the patterns from A2_address_standardization:
    - Uppercase
    - Remove periods
    - Standardize street types (Ave -> AVE, Street -> ST)
    - Remove extra whitespace
    - Remove unit/apartment numbers
    """
    if not address or pd.isna(address):
        return None
    
    addr = str(address).upper().strip()
    
    # Remove periods
    addr = addr.replace('.', '')
    
    # Standardize street types
    replacements = [
        (r'\bAVENUE\b', 'AVE'),
        (r'\bAV\b', 'AVE'),
        (r'\bSTREET\b', 'ST'),
        (r'\bBOULEVARD\b', 'BLVD'),
        (r'\bDRIVE\b', 'DR'),
        (r'\bCOURT\b', 'CT'),
        (r'\bPLACE\b', 'PL'),
        (r'\bLANE\b', 'LN'),
        (r'\bROAD\b', 'RD'),
        (r'\bWAY\b', 'WY'),
        (r'\bCIRCLE\b', 'CIR'),
        (r'\bTERRACE\b', 'TER'),
        (r'\bPARKWAY\b', 'PKWY'),
        (r'\bHIGHWAY\b', 'HWY'),
    ]
    
    for pattern, replacement in replacements:
        addr = re.sub(pattern, replacement, addr)
    
    # Convert spelled-out numbers to digits (for street names)
    number_words = {
        'FIRST': '1ST', 'SECOND': '2ND', 'THIRD': '3RD', 'FOURTH': '4TH',
        'FIFTH': '5TH', 'SIXTH': '6TH', 'SEVENTH': '7TH', 'EIGHTH': '8TH',
        'NINTH': '9TH', 'TENTH': '10TH', 'ELEVENTH': '11TH', 'TWELFTH': '12TH'
    }
    for word, num in number_words.items():
        addr = re.sub(rf'\b{word}\b', num, addr)
    
    # Remove unit/apartment designations
    addr = re.sub(r'\s+(APT|APARTMENT|UNIT|STE|SUITE|#)\s*\S*$', '', addr)
    
    # Remove extra whitespace
    addr = ' '.join(addr.split())
    
    return addr


def extract_address_from_name(name):
    """
    Try to extract an address from a project name.
    
    Many Gellerman entries are named by address (e.g., '2700 Shattuck Ave').
    """
    if not name:
        return None
    
    # Pattern: starts with a number, followed by street name
    addr_pattern = r'^(\d+\s+[A-Za-z0-9\s]+(?:ST|AVE|BLVD|DR|CT|PL|LN|RD|WY|WAY|STREET|AVENUE|BOULEVARD|DRIVE|COURT|PLACE|LANE|ROAD))'
    
    match = re.match(addr_pattern, name.upper())
    if match:
        return normalize_address(match.group(1))
    
    # If no match, try normalizing the whole name (might be just an address)
    if re.match(r'^\d+\s+', name):
        return normalize_address(name)
    
    return None

print('Address normalization functions loaded')

Address normalization functions loaded


## 5. Data Quality Validation

Validate that coordinates fall within Berkeley's boundaries.

In [8]:
# ============================================================================
# COORDINATE VALIDATION (Berkeley bounds from A3)
# ============================================================================

# Berkeley geographic bounds
BERKELEY_BOUNDS = {
    'lat_min': 37.845,
    'lat_max': 37.915,
    'lon_min': -122.325,
    'lon_max': -122.235
}

def validate_berkeley_coordinates(lat, lon):
    """
    Check if coordinates fall within Berkeley city bounds.
    
    Returns:
    --------
    bool
        True if within bounds, False otherwise
    """
    if pd.isna(lat) or pd.isna(lon):
        return False
    
    lat_ok = BERKELEY_BOUNDS['lat_min'] <= lat <= BERKELEY_BOUNDS['lat_max']
    lon_ok = BERKELEY_BOUNDS['lon_min'] <= lon <= BERKELEY_BOUNDS['lon_max']
    
    return lat_ok and lon_ok


def validate_dataframe_coordinates(df):
    """
    Add validation column to dataframe.
    """
    df = df.copy()
    df['coords_valid'] = df.apply(
        lambda row: validate_berkeley_coordinates(row.get('latitude'), row.get('longitude')),
        axis=1
    )
    return df

print(f'Berkeley bounds: {BERKELEY_BOUNDS}')
print('Coordinate validation functions loaded')

Berkeley bounds: {'lat_min': 37.845, 'lat_max': 37.915, 'lon_min': -122.325, 'lon_max': -122.235}
Coordinate validation functions loaded


## 6. Load and Parse KML Data

The Gellerman map uses a NetworkLink, so we need to fetch the actual KML data.

In [9]:
# ============================================================================
# FETCH KML DATA
# ============================================================================

timer_start('FETCHING GELLERMAN MAP DATA')

# Google Maps KML URL (from the NetworkLink)
GELLERMAN_KML_URL = 'https://www.google.com/maps/d/u/1/kml?forcekml=1&mid=1DAOhWMGII579mvz7sIghdMN-XJWSAOg'

# Local file path (backup)
LOCAL_KML_PATH = ROOT / 'data/reference/gellerman_berkeley_housing_full.kml'

kml_content = None
kml_source = None

# Try to fetch from Google Maps
if HAS_REQUESTS:
    print(f'Attempting to fetch KML from Google Maps...')
    print(f'URL: {GELLERMAN_KML_URL[:60]}...')
    
    try:
        response = requests.get(GELLERMAN_KML_URL, timeout=30)
        response.raise_for_status()
        
        kml_content = response.text
        kml_source = 'Google Maps API'
        
        print(f'SUCCESS: Fetched {len(kml_content):,} bytes')
        
        # Save a copy locally for future use
        LOCAL_KML_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(LOCAL_KML_PATH, 'w', encoding='utf-8') as f:
            f.write(kml_content)
        print(f'Saved local copy to: {LOCAL_KML_PATH}')
        
    except requests.exceptions.RequestException as e:
        print(f'Could not fetch from Google Maps: {e}')
        print('Will try local file...')

# Try local file if fetch failed
if kml_content is None and LOCAL_KML_PATH.exists():
    print(f'\nLoading from local file: {LOCAL_KML_PATH}')
    with open(LOCAL_KML_PATH, 'r', encoding='utf-8') as f:
        kml_content = f.read()
    kml_source = 'Local file'
    print(f'Loaded {len(kml_content):,} bytes')

# Check NetworkLink file (the original file might just be a reference)
if kml_content is None:
    orig_kml = ROOT / 'data/reference/gellerman_berkeley_housing.kml'
    if orig_kml.exists():
        print(f'\nChecking original KML file: {orig_kml}')
        with open(orig_kml, 'r', encoding='utf-8') as f:
            orig_content = f.read()
        
        # Check if it's a NetworkLink
        if '<NetworkLink>' in orig_content:
            print('Original file is a NetworkLink (reference only)')
            print('Please manually download the full KML from Google Maps')
        else:
            kml_content = orig_content
            kml_source = 'Original KML file'

if kml_content:
    print(f'\nKML Source: {kml_source}')
else:
    print('\nERROR: No KML data available')
    print('\nManual download instructions:')
    print('1. Go to: https://www.google.com/maps/d/viewer?mid=1DAOhWMGII579mvz7sIghdMN-XJWSAOg')
    print('2. Click the three dots menu (...)')
    print('3. Select "Download KML"')
    print('4. Save as: data/reference/gellerman_berkeley_housing_full.kml')

timer_end()

FETCHING GELLERMAN MAP DATA
Started: 2026-02-26 19:55:13
Attempting to fetch KML from Google Maps...
URL: https://www.google.com/maps/d/u/1/kml?forcekml=1&mid=1DAOhWM...
SUCCESS: Fetched 1,585,504 bytes
Saved local copy to: /Users/johngage/berkeley-data/data/reference/gellerman_berkeley_housing_full.kml

KML Source: Google Maps API
Completed: 2026-02-26 19:55:14
Duration: 0.99 seconds


In [10]:
# ============================================================================
# PARSE KML PLACEMARKS
# ============================================================================

timer_start('PARSING KML PLACEMARKS')

placemarks = []

if kml_content:
    print('Parsing KML content...')
    placemarks = parse_kml_placemarks(kml_content)
    
    print(f'\nExtracted {len(placemarks)} placemarks')
    
    if placemarks:
        # Show sample
        print('\nSample placemarks:')
        for i, pm in enumerate(placemarks[:5]):
            print(f'  {i+1}. {pm["name"]}')
            if pm['latitude'] and pm['longitude']:
                print(f'      Coords: ({pm["latitude"]:.6f}, {pm["longitude"]:.6f})')
            if pm['description']:
                desc_preview = pm['description'][:100].replace('\n', ' ')
                print(f'      Description: {desc_preview}...')
else:
    print('No KML content to parse')
    print('\nCreating sample data for demonstration...')
    
    # Sample data for testing (remove in production)
    placemarks = [
        {
            'name': '2700 Shattuck Ave',
            'description': '<a href="https://berkeleyside.org/2024/01/shattuck-project">News</a>',
            'latitude': 37.8598,
            'longitude': -122.2678
        },
        {
            'name': '1914 Fifth St',
            'description': '<a href="https://sfyimby.com/2023/05/fifth-street-berkeley">SFYimby</a>',
            'latitude': 37.8682,
            'longitude': -122.2993
        },
    ]
    print(f'Created {len(placemarks)} sample placemarks for testing')

timer_end()

PARSING KML PLACEMARKS
Started: 2026-02-26 19:55:14
Parsing KML content...

Extracted 207 placemarks

Sample placemarks:
  1. North Berkeley BART station
      Description: <img src="https://mymaps.usercontent.google.com/hostedimage/m/*/3AL_Y2X7R3VOzzSBn3Qi851jwHpXslhOkMbN...
  2. 1110 University Ave
      Coords: (37.868915, -122.291306)
      Description: <img src="https://mymaps.usercontent.google.com/hostedimage/m/*/3AL_Y2X5a67hWrlixaBspguGtET9GfkJ1slF...
  3. 2001 Fourth St
      Coords: (37.867349, -122.299266)
      Description: unnamed (1): <br>LINK 1: <br>LINK 2: <br>LINK 3: <br>UNITS: <br>STORIES: <br>HEIGHT: <br>SQUARE FT: ...
  4. 1367 University Ave
      Description: <img src="https://mymaps.usercontent.google.com/hostedimage/m/*/3AL_Y2X7TGXpQ_Mj0V6ALmE2n4KX0kVulNd_...
  5. 1498 University Ave
      Description: <img src="https://mymaps.usercontent.google.com/hostedimage/m/*/3AL_Y2X5wy9D5mXQdorapbnr7_XAQt8D4lCP...
Completed: 2026-02-26 19:55:14
Duration: 0.03 seconds


## 7. Create Raw Projects DataFrame

In [11]:
# ============================================================================
# CREATE PROJECTS DATAFRAME
# ============================================================================

timer_start('CREATING PROJECTS DATAFRAME')

if placemarks:
    # Create DataFrame
    df_gellerman = pd.DataFrame(placemarks)
    
    # Add normalized address
    df_gellerman['address_normalized'] = df_gellerman['name'].apply(extract_address_from_name)
    
    # Validate coordinates
    df_gellerman = validate_dataframe_coordinates(df_gellerman)
    
    # Count URLs in description
    df_gellerman['url_count'] = df_gellerman['description'].apply(
        lambda x: len(extract_urls_from_html(x)) if x else 0
    )
    
    print(f'\nDataFrame shape: {df_gellerman.shape}')
    print(f'\nColumns: {list(df_gellerman.columns)}')
    
    # Statistics
    print(f'\nStatistics:')
    print(f'  Total projects: {len(df_gellerman)}')
    print(f'  With coordinates: {df_gellerman["latitude"].notna().sum()}')
    print(f'  Valid Berkeley coords: {df_gellerman["coords_valid"].sum()}')
    print(f'  With normalized address: {df_gellerman["address_normalized"].notna().sum()}')
    print(f'  With news links: {(df_gellerman["url_count"] > 0).sum()}')
    print(f'  Total URLs found: {df_gellerman["url_count"].sum()}')
    
    # Show sample
    print('\nSample data:')
    display(df_gellerman[['name', 'address_normalized', 'latitude', 'longitude', 'coords_valid', 'url_count']].head(10))
else:
    df_gellerman = pd.DataFrame()
    print('No placemarks to process')

timer_end()

CREATING PROJECTS DATAFRAME
Started: 2026-02-26 19:55:14

DataFrame shape: (207, 7)

Columns: ['name', 'description', 'latitude', 'longitude', 'address_normalized', 'coords_valid', 'url_count']

Statistics:
  Total projects: 207
  With coordinates: 37
  Valid Berkeley coords: 37
  With normalized address: 183
  With news links: 205
  Total URLs found: 2024

Sample data:


,name,address_normalized,latitude,longitude,coords_valid,url_count
0,North Berkeley BART station,None,NaN,NaN,False,35
1,1110 University Ave,1110 UNIVERSITY AVE,37.868915,-122.291306,True,1
2,2001 Fourth St,2001 4TH ST,37.867349,-122.299266,True,0
3,1367 University Ave,1367 UNIVERSITY AVE,NaN,NaN,False,19
4,1498 University Ave,1498 UNIVERSITY AVE,NaN,NaN,False,7
5,1581 University Ave,1581 UNIVERSITY AVE,NaN,NaN,False,7
6,1598 University Ave,1598 UNIVERSITY AVE,NaN,NaN,False,37
7,1652 University Ave,1652 UNIVERSITY AVE,NaN,NaN,False,23
8,1698 University Ave,1698 UNIVERSITY AVE,NaN,NaN,False,13
9,1717 University Ave,1717 UNIVERSITY AVE,37.871743,-122.273375,True,13


Completed: 2026-02-26 19:55:14
Duration: 0.03 seconds


## 8. Extract News Links

In [12]:
# ============================================================================
# EXTRACT AND CATEGORIZE NEWS LINKS
# ============================================================================

timer_start('EXTRACTING NEWS LINKS')

if placemarks:
    df_news = extract_news_links(placemarks)
    
    print(f'Extracted {len(df_news)} URLs total')
    
    # Add normalized address
    df_news['address_normalized'] = df_news['project_name'].apply(extract_address_from_name)
    
    # Source breakdown
    print('\nURLs by source:')
    source_counts = df_news['source_category'].value_counts()
    for source, count in source_counts.items():
        print(f'  {source}: {count}')
    
    # News sources only (exclude city/permit links)
    news_sources = ['Berkeleyside', 'SFYimby', 'SF Chronicle', 'Daily Cal', 'SFGate', 'Mercury News', 'East Bay Times']
    df_news_only = df_news[df_news['source_category'].isin(news_sources)]
    
    print(f'\nNews articles only: {len(df_news_only)}')
    
    # Sample news links
    print('\nSample news links:')
    display(df_news_only[['project_name', 'source_category', 'url']].head(10))
else:
    df_news = pd.DataFrame()
    print('No placemarks to extract links from')

timer_end()

EXTRACTING NEWS LINKS
Started: 2026-02-26 19:55:14
Extracted 2024 URLs total

URLs by source:
  Other: 1583
  SFYimby: 267
  Berkeleyside: 96
  City of Berkeley: 57
  Daily Cal: 11
  SFGate: 5
  Mercury News: 2
  SF Chronicle: 1
  Google Maps: 1
  East Bay Times: 1

News articles only: 383

Sample news links:


,project_name,source_category,url
3,North Berkeley BART station,SFYimby,https://sfyimby.com/2024/02/pre-application-fi...
18,North Berkeley BART station,Berkeleyside,https://www.berkeleyside.org/2023/12/13/north-...
19,North Berkeley BART station,SFYimby,https://sfyimby.com/2023/09/renderings-reveale...
24,North Berkeley BART station,SFYimby,https://sfyimby.com/2025/09/modified-plans-for...
33,North Berkeley BART station,Berkeleyside,https://www.berkeleyside.org/2022/06/03/counci...
42,1367 University Ave,Berkeleyside,https://www.berkeleyside.org/2020/07/10/a-mini...
45,1367 University Ave,SFYimby,https://sfyimby.com/2022/01/design-review-pend...
50,1367 University Ave,Berkeleyside,https://www.berkeleyside.org/2023/09/27/suppor...
63,1581 University Ave,SFYimby,https://sfyimby.com/2024/01/preliminary-applic...
64,1581 University Ave,SFYimby,https://sfyimby.com/2024/06/permits-filed-for-...


Completed: 2026-02-26 19:55:14
Duration: 0.04 seconds


## 9. Data Quality Report

In [13]:
# ============================================================================
# DATA QUALITY REPORT
# ============================================================================

timer_start('GENERATING DATA QUALITY REPORT')

quality_issues = []

if len(df_gellerman) > 0:
    print('DATA QUALITY CHECKS')
    print('='*70)
    
    # 1. Missing coordinates
    missing_coords = df_gellerman['latitude'].isna().sum()
    if missing_coords > 0:
        quality_issues.append(f'{missing_coords} projects missing coordinates')
        print(f'[WARNING] {missing_coords} projects missing coordinates')
    else:
        print('[PASS] All projects have coordinates')
    
    # 2. Invalid coordinates (outside Berkeley)
    invalid_coords = (~df_gellerman['coords_valid']).sum()
    if invalid_coords > 0:
        quality_issues.append(f'{invalid_coords} projects with coordinates outside Berkeley')
        print(f'[WARNING] {invalid_coords} projects outside Berkeley bounds')
        
        # Show examples
        invalid_examples = df_gellerman[~df_gellerman['coords_valid']][['name', 'latitude', 'longitude']].head(3)
        print('  Examples:')
        for _, row in invalid_examples.iterrows():
            print(f'    - {row["name"]}: ({row["latitude"]}, {row["longitude"]})')
    else:
        print('[PASS] All coordinates within Berkeley bounds')
    
    # 3. Missing normalized addresses
    missing_addr = df_gellerman['address_normalized'].isna().sum()
    if missing_addr > 0:
        quality_issues.append(f'{missing_addr} projects without parseable address')
        print(f'[INFO] {missing_addr} projects without parseable address')
    else:
        print('[PASS] All project names contain parseable addresses')
    
    # 4. Duplicate addresses
    addr_counts = df_gellerman['address_normalized'].dropna().value_counts()
    duplicates = addr_counts[addr_counts > 1]
    if len(duplicates) > 0:
        quality_issues.append(f'{len(duplicates)} duplicate addresses found')
        print(f'[WARNING] {len(duplicates)} duplicate addresses:')
        for addr, count in duplicates.head(5).items():
            print(f'    - {addr}: {count} entries')
    else:
        print('[PASS] No duplicate addresses')
    
    # 5. Projects without news links
    no_news = (df_gellerman['url_count'] == 0).sum()
    print(f'[INFO] {no_news} projects without news links ({100*no_news/len(df_gellerman):.1f}%)')
    
    print('\n' + '='*70)
    print(f'Total issues: {len(quality_issues)}')
else:
    print('No data to validate')

timer_end()

GENERATING DATA QUALITY REPORT
Started: 2026-02-26 19:55:14
DATA QUALITY CHECKS
[WARNING] 170 projects missing coordinates
[WARNING] 170 projects outside Berkeley bounds
  Examples:
    - North Berkeley BART station: (nan, nan)
    - 1367 University Ave: (nan, nan)
    - 1498 University Ave: (nan, nan)
[INFO] 24 projects without parseable address
[WARNING] 3 duplicate addresses:
    - 3132 MARTIN LUTHER KING: 2 entries
    - 2747 SAN PABLO AVE: 2 entries
    - 2740 SAN PABLO AVE: 2 entries
[INFO] 2 projects without news links (1.0%)

Total issues: 4
Completed: 2026-02-26 19:55:14
Duration: 0.00 seconds


## 10. Export Data

In [14]:
# ============================================================================
# EXPORT DATA FILES
# ============================================================================

timer_start('EXPORTING DATA')

outputs_dir = ROOT / 'outputs'
outputs_dir.mkdir(exist_ok=True)

files_created = []

# 1. Export raw projects
if len(df_gellerman) > 0:
    raw_path = outputs_dir / 'gellerman_raw.csv'
    
    # Select columns for export (exclude large description field)
    export_cols = ['name', 'address_normalized', 'latitude', 'longitude', 'coords_valid', 'url_count']
    df_export = df_gellerman[export_cols].copy()
    
    df_export.to_csv(raw_path, index=False)
    files_created.append(('gellerman_raw.csv', len(df_export), 'All Gellerman projects'))
    print(f'Saved: {raw_path}')
    print(f'  {len(df_export)} rows')

# 2. Export news links
if len(df_news) > 0:
    news_path = outputs_dir / 'gellerman_news_links.csv'
    df_news.to_csv(news_path, index=False)
    files_created.append(('gellerman_news_links.csv', len(df_news), 'All extracted URLs'))
    print(f'\nSaved: {news_path}')
    print(f'  {len(df_news)} URLs')

# 3. Export summary
summary_path = outputs_dir / 'a6_import_summary.txt'
with open(summary_path, 'w') as f:
    f.write('A6 COMMUNITY MAP IMPORT SUMMARY\n')
    f.write(f'Generated: {datetime.now().isoformat()}\n')
    f.write(f'KML Source: {kml_source}\n')
    f.write('='*50 + '\n\n')
    
    f.write('PROJECT STATISTICS\n')
    f.write('-'*30 + '\n')
    if len(df_gellerman) > 0:
        f.write(f'Total projects: {len(df_gellerman)}\n')
        f.write(f'With coordinates: {df_gellerman["latitude"].notna().sum()}\n')
        f.write(f'Valid Berkeley coords: {df_gellerman["coords_valid"].sum()}\n')
        f.write(f'With normalized address: {df_gellerman["address_normalized"].notna().sum()}\n')
    
    f.write('\nNEWS LINK STATISTICS\n')
    f.write('-'*30 + '\n')
    if len(df_news) > 0:
        f.write(f'Total URLs extracted: {len(df_news)}\n')
        for source, count in df_news['source_category'].value_counts().items():
            f.write(f'  {source}: {count}\n')
    
    f.write('\nDATA QUALITY ISSUES\n')
    f.write('-'*30 + '\n')
    for issue in quality_issues:
        f.write(f'- {issue}\n')
    if not quality_issues:
        f.write('No critical issues found\n')

files_created.append(('a6_import_summary.txt', '-', 'Import statistics'))
print(f'\nSaved: {summary_path}')

# Summary table
print('\n' + '='*70)
print('FILES CREATED:')
print('-'*70)
for filename, rows, desc in files_created:
    print(f'  {filename:35} {str(rows):>8} rows  - {desc}')

timer_end()

EXPORTING DATA
Started: 2026-02-26 19:55:14
Saved: /Users/johngage/berkeley-data/outputs/gellerman_raw.csv
  207 rows

Saved: /Users/johngage/berkeley-data/outputs/gellerman_news_links.csv
  2024 URLs

Saved: /Users/johngage/berkeley-data/outputs/a6_import_summary.txt

FILES CREATED:
----------------------------------------------------------------------
  gellerman_raw.csv                        207 rows  - All Gellerman projects
  gellerman_news_links.csv                2024 rows  - All extracted URLs
  a6_import_summary.txt                      - rows  - Import statistics
Completed: 2026-02-26 19:55:14
Duration: 0.02 seconds


---

## Summary

This notebook:
- Fetched/parsed KML data from Eric Gellerman's Berkeley Development Map
- Extracted project names, coordinates, and descriptions
- Parsed HTML descriptions to extract news URLs
- Categorized URLs by source (Berkeleyside, SFYimby, etc.)
- Normalized addresses for matching with official data
- Validated coordinates against Berkeley boundaries

**Outputs:**
- `outputs/gellerman_raw.csv` - All projects
- `outputs/gellerman_news_links.csv` - News URLs with categories
- `outputs/a6_import_summary.txt` - Statistics

**Next:** Run `A7_comprehensive_integration.ipynb` to merge with official permit data.